In [1]:
# 🛠️ Instalação de bibliotecas (só por garantia)
!pip install pandas tqdm

import requests
import pandas as pd
from tqdm import tqdm

# 🧩 Função para coletar produtos via API VTEX (ajuste a URL base se necessário)
def coletar_produtos_vtex(query='pisos ceramicos', max_paginas=10):
    base_url = "https://www.obramax.com.br/api/catalog_system/pub/products/search"

    todos_produtos = []
    for pagina in tqdm(range(max_paginas), desc='Coletando páginas'):
        params = {
            "ft": query,
            "page": pagina
        }
        response = requests.get(base_url, params=params)
        if response.status_code != 200:
            print(f"Erro na página {pagina}: {response.status_code}")
            break

        produtos = response.json()
        if not produtos:
            break  # Sem mais produtos

        for p in produtos:
            item = p['items'][0]
            seller = item['sellers'][0]['commertialOffer']
            todos_produtos.append({
                'Nome': p['productName'],
                'Link': f"https://www.obramax.com.br/{p['linkText']}/p",
                'Preço': seller['Price'],
                'Disponibilidade': seller['AvailableQuantity'],
                'ID Produto': p['productId'],
                'Imagem': item['images'][0]['imageUrl'] if item['images'] else ''
            })

    return pd.DataFrame(todos_produtos)

# 🧪 Coleta os dados
df = coletar_produtos_vtex(query='pisos ceramicos', max_paginas=10)

# 💾 Salva em CSV
df.to_csv('/content/obramax_pisos.csv', index=False)
df.head()


Coletando páginas:   0%|          | 0/10 [00:00<?, ?it/s]

Erro na página 0: 400


""
